# AutoResearch Kaggle — Experiment Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Load results ---
if not os.path.exists("results.tsv"):
    print("No results.tsv found yet. Run a few experiments first!")
else:
    df = pd.read_csv("results.tsv", sep="\t")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["experiment"] = range(1, len(df) + 1)

    print(f"Total experiments: {len(df)}")
    print(f"Methods tried: {df['method'].nunique()} — {', '.join(df['method'].unique())}")
    print(f"Best Log Loss: {df['val_loss'].min():.6f}")
    print(f"Best Accuracy: {df['val_accuracy'].max():.6f}")
    print(f"Best method:   {df.loc[df['val_loss'].idxmin(), 'method']}")
    print()

    # --- Plot 1: Loss & Accuracy over time ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    colors = sns.color_palette("tab10", n_colors=df["method"].nunique())
    method_colors = dict(zip(df["method"].unique(), colors))

    for method, group in df.groupby("method"):
        ax1.plot(group["experiment"], group["val_loss"], marker="o", label=method,
                 color=method_colors[method], linewidth=2, markersize=5)
        ax2.plot(group["experiment"], group["val_accuracy"], marker="o", label=method,
                 color=method_colors[method], linewidth=2, markersize=5)

    # Best line
    best_loss = df["val_loss"].min()
    ax1.axhline(y=best_loss, color="red", linestyle="--", alpha=0.5, label=f"Best: {best_loss:.4f}")
    best_acc = df["val_accuracy"].max()
    ax2.axhline(y=best_acc, color="red", linestyle="--", alpha=0.5, label=f"Best: {best_acc:.4f}")

    ax1.set_title("Validation Log Loss (lower is better)", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Experiment #")
    ax1.set_ylabel("Log Loss")
    ax1.legend(fontsize=8)
    ax1.grid(True, linestyle="--", alpha=0.4)

    ax2.set_title("Validation Accuracy (higher is better)", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Experiment #")
    ax2.set_ylabel("Accuracy")
    ax2.legend(fontsize=8)
    ax2.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

    # --- Plot 2: Method comparison boxplot ---
    if df["method"].nunique() > 1:
        fig, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))
        df.boxplot(column="val_loss", by="method", ax=ax3)
        ax3.set_title("Log Loss by Method", fontsize=12, fontweight="bold")
        ax3.set_xlabel("")
        ax3.set_ylabel("Log Loss")
        plt.sca(ax3)
        plt.xticks(rotation=45, ha="right")

        df.boxplot(column="val_accuracy", by="method", ax=ax4)
        ax4.set_title("Accuracy by Method", fontsize=12, fontweight="bold")
        ax4.set_xlabel("")
        ax4.set_ylabel("Accuracy")
        plt.sca(ax4)
        plt.xticks(rotation=45, ha="right")

        plt.suptitle("")
        plt.tight_layout()
        plt.show()

    # --- Cumulative best ---
    fig, ax5 = plt.subplots(figsize=(10, 4))
    df["cumulative_best"] = df["val_loss"].cummin()
    ax5.step(df["experiment"], df["cumulative_best"], where="post", linewidth=2, color="#d62728")
    ax5.fill_between(df["experiment"], df["cumulative_best"], alpha=0.1, color="#d62728", step="post")
    ax5.set_title("Cumulative Best Log Loss", fontsize=12, fontweight="bold")
    ax5.set_xlabel("Experiment #")
    ax5.set_ylabel("Best Log Loss So Far")
    ax5.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    # --- Results table ---
    print("\n--- Full Experiment Log ---")
    display(df[["experiment", "timestamp", "method", "trials", "val_loss", "val_accuracy", "submission", "notes"]])